# SQL Readiness Hand-off

**Lesson dataset:** `dirty_data.csv`

This short hand-off lesson sits between Python table preparation and the Python-to-SQL bridge. It teaches what a prepared table needs to make its next step clear.

The worked examples use small dummy parcel, reading, delivery, returns, and bus-stop tables. Dummy data is not the lesson data. It only gives the operation somewhere neutral to run before you transfer the same thinking to `dirty_data.csv`.

## Setup: load the lesson table

A **library** is a packaged collection of tools. **pandas** is the Python library used here for table-shaped data such as CSV files. `import pandas as pd` loads that toolbox and gives it the short nickname `pd`; it does not load a dataset.

A **path** is the written route to a file. `Path` is a Python tool for storing that route without putting a personal computer location into the notebook. `df` is the variable name used for the lesson table.

| Part | Meaning |
|---|---|
| `Path` | A Python tool for holding a file path. |
| `dataset_path` | A label chosen to store the lesson file path. |
| `pd.read_csv(...)` | The pandas tool that reads comma-separated text into a table. |
| `df` | The label chosen to store the loaded lesson table. |
| `df.shape` | The table's row and column count. |

In [1]:
from pathlib import Path
import pandas as pd

dataset_path = Path("../../datasets/dirty_data.csv")
df = pd.read_csv(dataset_path)

print("Lesson dataset:", dataset_path.name)
print("Rows and columns:", df.shape)

Lesson dataset: dirty_data.csv
Rows and columns: (31, 5)


## Concept 1 — One row must have one clear meaning

A table's **grain** is what one row represents. SQL can query a table more safely when one row has one deliberate meaning, such as one parcel delivery or one workout record. `shape` reports the number of rows and columns so you can check the table's size before changing it.

### Command panel

`parcel_df.shape`

- `parcel_df` is the variable holding the dummy table.
- `.` reaches into that table.
- `shape` is a table attribute that reports `(rows, columns)`.

The worked example prints the rows first, then prints the shape.

In [2]:
parcel_df = pd.DataFrame({
    "parcel_id": ["P01", "P02", "P03"],
    "zone": ["North", "South", "North"],
    "weight_kg": [2.4, 1.1, 4.8]
})

print(parcel_df.to_string(index=False))
print("Rows and columns:", parcel_df.shape)

parcel_id   zone  weight_kg
      P01  North        2.4
      P02  South        1.1
      P03  North        4.8
Rows and columns: (3, 3)


**Question on the lesson dataset — `dirty_data.csv`**

Use `df.shape` to report the current number of rows and columns. Then inspect the repeated `Date` value in the lesson table and write one sentence explaining what you think one row represents. Do not remove anything yet.

## Concept 2 — Column names should survive the move into SQL

A **column name** labels one field in every row. SQL-friendly names are lowercase, consistent, and easy to type. Avoid spaces, brackets, and punctuation that make later queries harder to read.

### Command panel

`reading_df = reading_df.rename(columns={"Reading Date": "reading_date"})`

- `reading_df` is the label holding the table.
- `rename` is the pandas table tool for changing labels.
- `columns` says that the labels being changed are column names.
- `{old: new}` is a dictionary: a pair of labels saying which old name becomes which new name.
- `=` stores the changed table back under the same label.

In [3]:
reading_df = pd.DataFrame({
    "Reading Date": ["2026-03-01", "2026-03-02"],
    "Room Temp (C)": [19.5, 20.1]
})
reading_df = reading_df.rename(columns={
    "Reading Date": "reading_date",
    "Room Temp (C)": "room_temp_c"
})

print(reading_df.columns.tolist())

['reading_date', 'room_temp_c']


**Question on the lesson dataset — `dirty_data.csv`**

Rename the lesson columns exactly as follows: `Duration` → `duration_minutes`, `Date` → `activity_date`, `Pulse` → `pulse`, `Maxpulse` → `max_pulse`, and `Calories` → `calories`. Print `df.columns.tolist()` so you can check the result.

## Concept 3 — Types tell the next tool what a value is

A **data type** describes what kind of value a column contains. Dates should behave like dates, and measurements should behave like numbers. Correct types make sorting, filtering, grouping, and SQL comparisons deliberate instead of accidental.

### Command panel

`delivery_df["delivery_date"] = pd.to_datetime(delivery_df["delivery_date"])`

- `delivery_df["delivery_date"]` selects one named column.
- `pd.to_datetime(...)` turns date-looking text into date values.
- The assignment stores the converted column back in the table.
- `delivery_df.dtypes` reports the types pandas is using.

In [4]:
delivery_df = pd.DataFrame({
    "delivery_date": ["2026-03-01", "2026-03-02"],
    "packages": ["12", "7"]
})
delivery_df["delivery_date"] = pd.to_datetime(delivery_df["delivery_date"])
delivery_df["packages"] = pd.to_numeric(delivery_df["packages"])

print(delivery_df.dtypes)

delivery_date    datetime64[ns]
packages                  int64
dtype: object


**Question on the lesson dataset — `dirty_data.csv`**

Convert `activity_date` to a date type and convert `duration_minutes`, `pulse`, `max_pulse`, and `calories` to numeric types. Then print `df.dtypes`. If a value cannot convert cleanly, inspect it before deciding what to do.

## Concept 4 — Missing values and duplicates need decisions

A **missing value** is an empty or unavailable cell. A **duplicate row** is a repeated full record. Neither should be removed automatically: a missing value may need a careful replacement, and a duplicate may be a mistake or a legitimate repeated event.

### Command panel

- `returns_df.isna().sum()` counts missing cells in each column.
- `returns_df.duplicated().sum()` counts repeated full rows.
- `drop_duplicates()` removes repeated full rows only when you have decided they are accidental.
- `fillna(...)` replaces missing values with a value you have justified.

In [5]:
returns_df = pd.DataFrame({
    "return_id": [101, 102, 102],
    "reason": ["size", None, None],
    "refund_amount": [12.0, 8.5, 8.5]
})

print("Missing values:")
print(returns_df.isna().sum())
print("Duplicate rows:", returns_df.duplicated().sum())

Missing values:
return_id        0
reason           1
refund_amount    0
dtype: int64
Duplicate rows: 1


**Question on the lesson dataset — `dirty_data.csv`**

Inspect `df.isna().sum()` and `df.duplicated().sum()`. Decide whether the repeated `Date` row is accidental. Remove only accidental duplicates, make a deliberate decision about the missing `calories` value, and print both checks again. State your reason in a markdown note after the code.

## Concept 5 — `prepared_df` is the hand-off contract

A **contract** is an agreed name and shape that the next step can rely on. The bridge checks for a pandas table named `prepared_df`. `prepared_df` is not a special pandas word; it is the exact variable name agreed between this Python preparation step and the SQL bridge.

A useful SQL-ready proof includes: one clear row meaning, SQL-friendly column names, deliberate types, no unexplained missing cells, no accidental duplicates, and a table that can be inspected with `shape`, `columns`, and `head()`.

In [6]:
prepared_df = parcel_df.copy()
print("prepared_df columns:", prepared_df.columns.tolist())
print("Rows and columns:", prepared_df.shape)
print("Missing cells:", int(prepared_df.isna().sum().sum()))

prepared_df columns: ['parcel_id', 'zone', 'weight_kg']
Rows and columns: (3, 3)
Missing cells: 0


**Question on the lesson dataset — `dirty_data.csv`**

After your earlier decisions, assign the prepared lesson table to the exact name `prepared_df`. Print `prepared_df.shape`, `prepared_df.columns.tolist()`, `prepared_df.dtypes`, `prepared_df.isna().sum()`, and `prepared_df.head()`. Then write three plain-English sentences explaining why this table is ready for a SQL query.

## Concept 6 — A browser can hold CSV text instead of a file path

In a local notebook, `pd.read_csv(dataset_path)` can read a file from the supplied `datasets` folder. In the browser bridge, the CSV content is held in a text variable named `csv_text`.

`StringIO` is a Python tool from the `io` module. It wraps text so a tool that expects a file-like object can read that text as if it were a file. It does not clean, rename, or analyse the data. It only changes how the text reaches `read_csv`.

### Command panel

`prepared_df = pd.read_csv(StringIO(csv_text))`

- `csv_text` is a string variable containing CSV characters.
- `StringIO(csv_text)` wraps those characters in a file-like text object.
- `pd.read_csv(...)` reads that object into a pandas table.
- `prepared_df =` stores the result under the bridge's required name.

In [7]:
from io import StringIO

csv_text = "stop_name,arrivals\nCentral,12\nHarbour,8"
prepared_df = pd.read_csv(StringIO(csv_text))
print(prepared_df.to_string(index=False))

stop_name  arrivals
  Central        12
 Harbour         8


**Question on the lesson dataset — `dirty_data.csv`**

Write a plain-English answer explaining why the browser bridge uses `StringIO(csv_text)` instead of the local file-path form `pd.read_csv(dataset_path)`. Include the exact final table name the bridge expects: `prepared_df`.

## Before you open the bridge

Say these decisions aloud before moving on:

1. What does one row mean?
2. Are the column names easy to query?
3. Are the value types deliberate?
4. What happened to missing values and duplicates?
5. Is the final table named `prepared_df`?

The next lesson uses the same idea in the browser: a small visible Python load step creates `prepared_df`, then SQL is allowed to query the prepared table.